# Custom RAG with Ollama Models

This notebook tests a custom RAG setup using:
- **OpenAI Embeddings** (text-embedding-3-small) for retrieval
- **Ollama Models** (local LLMs) for generation
- **Ragas** for evaluation

Benefits:
- Full control over RAG pipeline
- Use local Ollama models (cost-effective)
- Easy to swap embedding models or generators

## 1. Setup and Imports

In [ ]:
# Setup path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

import pandas as pd
from custom_rag_client import CustomRAGClient
from custom_evaluation_pipeline import CustomEvaluationPipeline
from utils import load_dataset_from_jsonl

print("✓ Imports successful")

## 2. Configuration

Configure your Ollama and embedding models here.

In [ ]:
# Configuration
CONFIG = {
    # Ollama settings
    'ollama_base_url': 'http://localhost:11434',
    'ollama_model': 'llama3.1:8b',  # Change to your preferred model
    
    # Embedding settings
    'embedding_model': 'text-embedding-3-small',  # OpenAI embedding
    
    # RAG settings
    'top_k': 5,  # Number of documents to retrieve
    'temperature': 0.1,  # Generation temperature
    
    # Evaluation settings
    'evaluator_model': 'gpt-4o-mini',  # Model for Ragas evaluation
}

# Available Ollama models (add more as needed)
AVAILABLE_OLLAMA_MODELS = [
    'llama3.1:8b',
    'llama3.1:70b',
    'mistral:7b',
    'mixtral:8x7b',
    'phi3:mini',
    'gemma2:9b',
]

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Initialize Custom RAG Client

We'll create a RAG client and load documents into it.

In [ ]:
# Initialize custom RAG client
client = CustomRAGClient(
    ollama_base_url=CONFIG['ollama_base_url'],
    ollama_model=CONFIG['ollama_model'],
    embedding_model=CONFIG['embedding_model'],
    top_k=CONFIG['top_k']
)

print("✓ Custom RAG client initialized")
print(f"  Generator: {client.ollama_model}")
print(f"  Embeddings: {client.embedding_model}")

## 4. Load Documents

Load your knowledge base documents. You can load from:
- A text file
- A JSON file
- A directory of files
- Or add documents directly

In [ ]:
# Option 1: Load from a directory or file
# documents_path = Path.cwd().parent.parent / 'material'  # Adjust path to your documents
# client.load_documents(str(documents_path))

# Option 2: Add documents manually (for testing)
sample_documents = [
    "AccessMatrix is a comprehensive identity and access management solution.",
    "The UAM module handles user authentication and authorization.",
    "UAS provides authentication services including OTP and biometric authentication.",
    "AccessMatrix supports SAML, OAuth, and OpenID Connect protocols.",
    "The system can be deployed on-premises or in the cloud.",
    "Multi-factor authentication (MFA) is supported through various methods.",
    "AccessMatrix integrates with HSM for secure key management.",
    "The administration guide covers system configuration and management.",
]

client.add_documents(sample_documents)

print(f"✓ Loaded {len(client.documents)} documents")
print(f"✓ Created {len(client.document_embeddings)} embeddings")

## 5. Test RAG System

Test the RAG with a sample query to ensure everything works.

In [ ]:
# Test connection to Ollama
print("Testing Ollama connection...")
if client.test_connection():
    print("✓ Ollama connection successful")
else:
    print("✗ Ollama connection failed - check if Ollama is running")

# Test RAG query
print("\nTesting RAG query...")
test_result = client.query("What is AccessMatrix?")

if test_result['success']:
    print("✓ RAG query successful\n")
    print("Answer:")
    print(test_result['answer'])
    print(f"\nRetrieved {len(test_result['contexts'])} contexts")
else:
    print(f"✗ Query failed: {test_result['error']}")

## 6. Load Test Dataset

In [ ]:
# Load test dataset
dataset_path = Path.cwd().parent / 'data' / 'test_qa_pairs.jsonl'

if dataset_path.exists():
    dataset = load_dataset_from_jsonl(str(dataset_path))
    print(f"Loaded {len(dataset)} test questions")
    
    print("\nFirst 3 questions:")
    for i, item in enumerate(dataset[:3]):
        print(f"{i+1}. {item['user_input']}")
else:
    print(f"⚠️  Test dataset not found at {dataset_path}")
    print("Creating a sample dataset for testing...")
    
    # Create sample dataset
    dataset = [
        {
            'user_input': 'What is AccessMatrix?',
            'reference': 'AccessMatrix is a comprehensive identity and access management solution.',
            'reference_contexts': ['AccessMatrix is a comprehensive identity and access management solution.']
        },
        {
            'user_input': 'What authentication methods are supported?',
            'reference': 'AccessMatrix supports multi-factor authentication including OTP and biometric authentication.',
            'reference_contexts': ['UAS provides authentication services including OTP and biometric authentication.', 'Multi-factor authentication (MFA) is supported through various methods.']
        },
        {
            'user_input': 'Which protocols does AccessMatrix support?',
            'reference': 'AccessMatrix supports SAML, OAuth, and OpenID Connect protocols.',
            'reference_contexts': ['AccessMatrix supports SAML, OAuth, and OpenID Connect protocols.']
        },
    ]
    print(f"Created {len(dataset)} sample questions")

## 7. Initialize Evaluation Pipeline

In [ ]:
# Initialize evaluation pipeline
pipeline = CustomEvaluationPipeline(
    rag_client=client,
    evaluator_model=CONFIG['evaluator_model']
)

print("✓ Evaluation pipeline initialized")
print(f"✓ Ragas evaluator: {CONFIG['evaluator_model']}")

## 8. Run Evaluation

In [ ]:
# Run evaluation
print("\n" + "="*70)
print("STARTING EVALUATION")
print("="*70)

result = pipeline.run_evaluation(
    test_dataset=dataset,
    model_name=f"custom_rag_{CONFIG['ollama_model'].replace(':', '_')}",
    verbose=True,
    temperature=CONFIG['temperature']
)

## 9. Analyze Results

In [ ]:
# Check if evaluation succeeded
if result['success']:
    print("✅ Evaluation completed successfully!\n")
    
    # Get results as DataFrame
    results_df = result['ragas_results'].to_pandas()
    
    # Display detailed results
    print("\nDetailed Results (per question):")
    print("=" * 80)
    print(results_df.to_string())
    print("=" * 80)
    
    # Summary statistics
    print("\nSummary Statistics:")
    print("=" * 80)
    summary = results_df.mean().to_frame(name='Mean')
    summary['Std'] = results_df.std()
    summary['Min'] = results_df.min()
    summary['Max'] = results_df.max()
    print(summary.to_string())
    print("=" * 80)
    
else:
    print("✗ Evaluation failed!")
    if 'error' in result:
        print(f"Error: {result['error']}")

# Show failed queries if any
if result['failed_queries']:
    print(f"\n⚠️  {len(result['failed_queries'])} queries failed:")
    for i, failed in enumerate(result['failed_queries'][:3]):
        print(f"{i+1}. {failed['question'][:60]}...")
        print(f"   Error: {failed['error']}")

## 10. Inspect Individual Responses

In [ ]:
# Look at a few examples
if result['success'] and result['eval_data']:
    print("\nExample Responses:\n")
    
    for i, item in enumerate(result['eval_data'][:3]):
        print("=" * 80)
        print(f"Question {i+1}: {item['user_input']}")
        print(f"\nGenerated Answer:\n{item['response']}")
        print(f"\nGround Truth:\n{item['reference']}")
        print(f"\nRetrieved Contexts: {len(item['retrieved_contexts'])}")
        if item['retrieved_contexts']:
            print("Contexts:")
            for j, ctx in enumerate(item['retrieved_contexts'][:2]):
                print(f"  {j+1}. {ctx[:150]}...")
        print("=" * 80 + "\n")

## 11. Visualize Metrics

In [ ]:
import plotly.graph_objects as go

if result['success']:
    # Create bar chart of average metrics
    metrics = results_df.mean()
    
    fig = go.Figure(data=[
        go.Bar(
            x=metrics.index,
            y=metrics.values,
            text=[f"{v:.3f}" for v in metrics.values],
            textposition='auto',
        )
    ])
    
    fig.update_layout(
        title=f"Custom RAG Performance: {CONFIG['ollama_model']} + {CONFIG['embedding_model']}",
        xaxis_title="Metric",
        yaxis_title="Score",
        yaxis_range=[0, 1],
        height=500
    )
    
    fig.show()

## 12. Save Results

In [ ]:
from datetime import datetime
from utils import save_results

if result['success']:
    # Create results directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_name = CONFIG['ollama_model'].replace(':', '_')
    results_dir = Path.cwd().parent / 'results' / f'custom_rag_{model_name}_{timestamp}'
    
    # Save
    save_results(
        results=result['ragas_results'],
        model_name=result['model_name'],
        output_dir=str(results_dir)
    )
    
    print(f"\n✅ Results saved to: {results_dir}")

## 13. (Optional) Test Multiple Ollama Models

Uncomment to test multiple Ollama models and compare their performance.

In [ ]:
# # Select models to test (make sure they're installed in Ollama)
# models_to_test = [
#     'llama3.1:8b',
#     'mistral:7b',
#     'phi3:mini',
# ]

# # Run multi-model evaluation
# multi_results = pipeline.run_multi_model_evaluation(
#     test_dataset=dataset,
#     ollama_models=models_to_test,
#     temperature=CONFIG['temperature']
# )

# # Compare results
# comparison_data = []
# for model, res in multi_results.items():
#     if res['success']:
#         metrics = res['ragas_results'].to_pandas().mean()
#         comparison_data.append({
#             'model': model,
#             **metrics.to_dict()
#         })

# comparison_df = pd.DataFrame(comparison_data)
# print("\nModel Comparison:")
# print(comparison_df.to_string())

## Summary

✅ This notebook demonstrates:
1. Custom RAG implementation with full control
2. Using Ollama models for local generation
3. OpenAI embeddings for retrieval
4. Ragas evaluation framework
5. Easy model switching and comparison

**Next Steps:**
- Load your actual document corpus
- Test with different Ollama models
- Experiment with different embedding models
- Compare against RAGFlow results